# 프로파일 파이프라인 단계별 스냅샷 뷰어

`_DEBUG_PROFILE_TRACE=True`로 실행 후 `C:\tmp\bdt_trace\session_*`에 생성된 pickle을 순서대로 로드하여 각 Stage의 DataFrame 변화를 시각화합니다.

## 사용 순서
1. BDT를 Python 모듈로 임포트하거나 GUI를 띄우기 전에 플래그를 켭니다:
   ```python
   import DataTool_dev_code.DataTool_optRCD_proto_ as bdt
   bdt._DEBUG_PROFILE_TRACE = True
   ```
2. GUI에서 **프로파일 분석**을 1회 실행 (사이클 1~2개 정도로 작게).
3. 아래 셀을 순서대로 실행.

In [ ]:
import os, glob, pickle
import pandas as pd
import matplotlib.pyplot as plt

TRACE_ROOT = r"C:\tmp\bdt_trace"
sessions = sorted(
    [d for d in glob.glob(os.path.join(TRACE_ROOT, "session_*")) if os.path.isdir(d)],
    key=os.path.getmtime, reverse=True)
print("세션 목록:")
for i, s in enumerate(sessions):
    print(f"  [{i}] {os.path.basename(s)}")

SESSION = sessions[0] if sessions else None
print(f"\n선택된 세션: {SESSION}")

In [ ]:
# 세션 내 모든 pickle 로드 (번호 순)
assert SESSION, "세션이 없습니다. 먼저 _DEBUG_PROFILE_TRACE=True 로 앱 실행"
files = sorted(glob.glob(os.path.join(SESSION, "*.pkl")))
snaps = []
for f in files:
    with open(f, "rb") as fh:
        p = pickle.load(fh)
    p["_file"] = os.path.basename(f)
    snaps.append(p)

# 요약 테이블
summary = pd.DataFrame([
    {
        "#": i,
        "stage": s["stage"],
        "tag": s["tag"],
        "shape": s.get("shape"),
        "n_cols": len(s.get("columns", [])) if s.get("columns") else 0,
    }
    for i, s in enumerate(snaps)
])
summary

## 각 단계 컬럼 / dtype 변화

In [ ]:
for s in snaps:
    print(f"── {s['stage']} [{s['tag']}] shape={s.get('shape')} ──")
    cols = s.get("columns") or []
    dtypes = s.get("dtypes") or {}
    for c in cols:
        print(f"  {c:<22} {dtypes.get(c, '?')}")
    print()

## 특정 Stage의 head 보기

In [ ]:
IDX = 0  # 확인할 스냅샷 번호
s = snaps[IDX]
print(f"[{IDX}] {s['stage']} / {s['tag']} — shape={s.get('shape')}")
s.get("head")

## SOC/전압 궤적 비교 (Stage 6, 7)
누적 ChgCap/DchgCap가 REST/반대 condition 행까지 ffill되어 SOC가 연속적으로 변하는지 확인.

In [ ]:
calc_snap = next((s for s in snaps if s["stage"] == "S6_calc_axis"), None)
if calc_snap is not None:
    df = calc_snap["df"]
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    axes[0].plot(df["TimeMin"], df["Voltage"], lw=0.5)
    axes[0].set_ylabel("V")
    axes[1].plot(df["TimeMin"], df["ChgCap"], label="ChgCap (ffill)", lw=0.5)
    axes[1].plot(df["TimeMin"], df["DchgCap"], label="DchgCap (ffill)", lw=0.5)
    axes[1].set_ylabel("Cap (0~1)"); axes[1].legend()
    axes[2].plot(df["TimeMin"], df["SOC"], lw=0.5, color="red")
    axes[2].set_ylabel("SOC"); axes[2].set_xlabel("Time(min)")
    plt.tight_layout(); plt.show()
else:
    print("S6_calc_axis 스냅샷이 없습니다.")

## OCV/CCV - SOC 분포 (Stage 7)

In [ ]:
cf = next((s for s in snaps if s["stage"] == "S7_output_cycfile_soc"), None)
if cf is not None:
    cfdf = cf["df"]
    print(f"cycfile_soc shape={cfdf.shape}")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(cfdf["SOC"], cfdf["OCV"], s=8, label="OCV")
    ax.scatter(cfdf["SOC"], cfdf["CCV"], s=8, label="CCV")
    ax.set_xlabel("SOC (%)"); ax.set_ylabel("Voltage (V)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.show()
else:
    print("S7_output_cycfile_soc 없음 (OCV/CCV 데이터 없는 실험)")

## 단계별 행 수 감소 그래프

In [ ]:
row_counts = [(s["stage"], s.get("shape", (0, 0))[0]) for s in snaps]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(row_counts)), [r[1] for r in row_counts])
ax.set_xticks(range(len(row_counts)))
ax.set_xticklabels([r[0] for r in row_counts], rotation=45, ha="right")
ax.set_ylabel("rows"); ax.set_title("Stage별 행 수 변화")
plt.tight_layout(); plt.show()